# SWOT 분석 (데이터 기반)

## 목적
이 노트북은 내부 데이터(성과·운영·고객)와 외부 데이터(시장·경쟁·트렌드)를 결합해 **강점(S)·약점(W)·기회(O)·위협(T)** 요소를 정량/정성으로 도출하고, 최종적으로 **TOWS 전략(SO/ST/WO/WT)**까지 제안하는 것을 목표로 합니다.

---

## 분석 흐름도

```
┌─────────────────────────────────────────────────────────────┐
│                    데이터 수집 (CSV)                          │
│  내부: 매출, 운영, 고객만족도, 리뷰    외부: 시장, 경쟁, 트렌드  │
└────────────────────┬────────────────────────────────────────┘
                     ▼
┌─────────────────────────────────────────────────────────────┐
│              Step 1~3: 환경 설정 및 데이터 준비                │
│  라이브러리 설치 → Import → 경로 설정 → 헬퍼 함수 정의         │
└────────────────────┬────────────────────────────────────────┘
                     ▼
┌──────────────────────────┐  ┌───────────────────────────────┐
│  내부 분석 (S / W 도출)    │  │   외부 분석 (O / T 도출)       │
│                          │  │                               │
│  Step 4: 판매 실적 분석    │  │  Step 6: 시장 데이터 분석      │
│   → 산업 대비 상대 성장률   │  │   → 매력도-경쟁 강도 매트릭스   │
│                          │  │                               │
│  Step 5: 고객 만족도 분석  │  │  Step 7: 소비자 트렌드 분석    │
│   → IPA(중요도-성과) 분석   │  │   → 영향력-관련성 매트릭스     │
│                          │  │                               │
│  Step 8: 리뷰 텍스트 마이닝 │  │                               │
│   → TF-IDF + NMF 토픽추출  │  │                               │
└────────────┬─────────────┘  └──────────────┬────────────────┘
             └──────────┬────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────────┐
│          Step 9: SWOT 매트릭스 통합 (계량화)                  │
│   S/W/O/T 각 요소에 정량 점수 부여 → 4분할 막대 그래프         │
└────────────────────┬────────────────────────────────────────┘
                     ▼
┌─────────────────────────────────────────────────────────────┐
│        Step 10: TOWS 전략 생성 (OpenAI GPT 활용)             │
│   MinMax 정규화 → S×O/S×T/W×O/W×T 조합 → LLM 전략 제안      │
└────────────────────┬────────────────────────────────────────┘
                     ▼
┌─────────────────────────────────────────────────────────────┐
│           Step 11: 최종 결과 시각화 및 요약                    │
│   전략적 위치(사분면) + 우선순위 TOP 5 전략 출력               │
└─────────────────────────────────────────────────────────────┘
```

---

## 사용 데이터 (CSV)

| 구분 | 파일명 | 용도 | 분석 대상 |
|------|--------|------|----------|
| 내부 | `sales_data.csv` | 매출/수익 지표 | S / W |
| 내부 | `operational_metrics.csv` | 운영 효율/품질 지표 | S / W |
| 내부 | `customer_satisfaction.csv` | 고객만족/불만 지표 | S / W |
| 내부 | `customer_reviews.csv` | 고객 리뷰 텍스트(+rating) | S / W |
| 외부 | `market_data.csv` | 시장 규모/점유 등 | O / T |
| 외부 | `industry_growth.csv` | 산업 성장률 | 비교 기준 |
| 외부 | `competitor_analysis.csv` | 경쟁사 비교 | O / T |
| 외부 | `industry_satisfaction.csv` | 산업/시장 만족도 | 비교 기준 |
| 외부 | `consumer_trends.csv` | 트렌드 영향/확산 | O / T |
| 외부 | `website_analytics.csv` | 유입/전환 디지털 지표 | 참고 |

## 주요 분석 기법

| 기법 | 적용 단계 | 설명 |
|------|----------|------|
| **상대 성장률** | Step 4 | 자사 매출 성장률 − 산업 평균 성장률 |
| **IPA 분석** | Step 5 | 중요도(Importance) × 성과(Performance) 매트릭스 |
| **TF-IDF + NMF** | Step 8 | 리뷰 텍스트에서 핵심 토픽(강점/약점 키워드) 자동 추출 |
| **MinMax 정규화** | Step 10 | 서로 다른 척도의 점수를 0~1로 통일 |
| **OpenAI GPT** | Step 10 | SWOT 조합별 실행 가능한 전략 문장 자동 생성 |

## 참고
- 로컬 실행 시 데이터는 `swot/` 폴더에서 로드합니다.
- OpenAI를 쓰려면 환경변수 `OPENAI_API_KEY`가 필요합니다(없어도 비-LLM 부분은 실행 가능).

## Step 1: 환경 설정 (Colab / 로컬 공통)

| 항목 | 설명 |
|------|------|
| **목적** | Colab 또는 로컬 환경에 맞춰 필요 라이브러리를 설치하고 한글 폰트를 설정합니다 |
| **실행 조건** | Colab → 자동 설치 / 로컬 → `requirements.txt`로 사전 설치 필요 |
| **주요 작업** | 패키지 설치, 한글 폰트(Nanum) 설치, Matplotlib 캐시 초기화 |

> 이 셀은 분석 환경을 준비하는 단계입니다. **로컬에서는 대부분 건너뜁니다.**

In [ ]:
# 환경 설정 (Colab/로컬 공통)
# - 로컬(macOS/Windows): 이 셀은 설치를 건너뜁니다. 터미널에서 아래를 실행하세요.
#   - Windows: .venv\Scripts\python -m pip install -r swot\requirements.txt
#   - macOS/Linux: .venv/bin/python -m pip install -r swot/requirements.txt
# - Colab: 필요한 경우에만 자동 설치/폰트 설치를 수행합니다.

import os
import sys
import shutil
import subprocess
from pathlib import Path

# Matplotlib 캐시 디렉터리를 작업폴더로 고정(권장: 권한 이슈/속도 개선)
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = _in_colab()

if IN_COLAB:
    print("Colab 환경: 라이브러리 설치를 진행합니다...")

    req = Path("requirements.txt")
    if not req.exists():
        alt = Path("swot") / "requirements.txt"
        req = alt if alt.exists() else req

    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    else:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "konlpy",
            "adjustText",
            "pandas",
            "seaborn",
            "matplotlib",
            "scikit-learn",
            "openpyxl",
            "numpy",
            "requests",
            "openai",
            "python-dotenv",
        ])

    # Colab에서 Nanum 폰트 설치 (로컬에서는 불필요)
    try:
        subprocess.check_call(
            [
                "bash",
                "-lc",
                "apt-get update -qq && apt-get install -y -qq fonts-nanum && fc-cache -fv >/dev/null",
            ]
        )
    except Exception as e:
        print(f"폰트 설치를 건너뜁니다: {e}")

    # Matplotlib 폰트 캐시 초기화
    try:
        shutil.rmtree(Path.home() / ".cache" / "matplotlib", ignore_errors=True)
    except Exception:
        pass

    print("Colab: 설치/폰트 설정 완료")
else:
    print("로컬 환경: 설치 셀을 건너뜁니다. 필요하면 requirements.txt를 설치하세요.")

### [설명] 환경 설정 및 라이브러리 준비
이 셀에서는 SWOT 분석에 필요한 파이썬 라이브러리와 환경을 설정합니다. Colab과 로컬 환경(macOS/Windows) 모두에서 실행할 수 있도록 분기 처리되어 있습니다. 
- Colab에서는 필요한 패키지를 자동 설치하고, 한글 폰트도 추가로 설치합니다.
- 로컬 환경에서는 requirements.txt 파일을 설치하도록 안내합니다.
- Matplotlib의 캐시 디렉토리를 작업 폴더로 고정하여 권한 문제를 방지합니다.

이 과정을 통해 이후 데이터 분석 및 시각화에 필요한 환경이 준비됩니다.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 로드 (swot/ 폴더 또는 현재 디렉터리에서 검색)
for env_path in [Path('swot/.env'), Path('.env')]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env 파일 로드 완료: {env_path}")
        break

# Matplotlib 캐시 디렉터리를 작업폴더로 고정(권한 이슈/속도 개선)
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from konlpy.tag import Okt
from adjustText import adjust_text
import re
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from itertools import product
import requests  # 사용 안 함
from openai import OpenAI  # OpenAI 라이브러리 import
import time  # API 호출 간격 제어용
import warnings

# 경고 메시지 무시 (선택 사항)
warnings.filterwarnings('ignore')

# Matplotlib 기본 설정 (크로스플랫폼 폰트 fallback)
plt.rcParams["font.family"] = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 Import 및 Matplotlib 설정 완료.")


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


# --- OpenAI API 키 설정 ---
# 1) 로컬/서버 공통: 환경변수 OPENAI_API_KEY (.env 또는 시스템 환경변수)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# 2) Colab일 때만: userdata(비밀번호)에서 가져오기 시도
if (OPENAI_API_KEY is None) and _in_colab():
    try:
        from google.colab import userdata  # type: ignore

        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
        print("OpenAI API 키 로드 완료 (Colab 비밀번호 사용).")
    except Exception as e:
        print(f"Colab userdata에서 API 키를 가져오지 못했습니다: {e}")

if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI 클라이언트 초기화 완료.")
else:
    openai_client = None
    print("경고: OPENAI_API_KEY가 설정되지 않았습니다. OpenAI 호출 셀은 실패할 수 있습니다.")

### [설명] 데이터 경로 및 파일 확인 함수
이 셀에서는 분석에 사용할 데이터 파일의 경로를 설정하고, 파일이 존재하는지 확인하는 함수를 정의합니다.
- 데이터는 기본적으로 `swot/` 폴더에 저장되어 있으며, Colab 환경에서는 경로를 변경할 수 있습니다.
- `check_file()` 함수는 파일의 존재 여부를 확인하고, 없을 경우 경고 메시지를 출력합니다.

이 과정을 통해 이후 분석에 필요한 데이터 파일이 올바르게 준비되어 있는지 점검할 수 있습니다.

In [ ]:
# --- 데이터 파일 경로 설정 ---
# 로컬 실행(권장): 저장소의 swot/ 폴더를 기본 데이터 경로로 사용
# Colab/Drive 사용 시: '/content/drive/MyDrive/...' 형태로 data_path를 바꿔주세요.

from pathlib import Path

if Path('swot').exists():
    data_path = Path('swot')
else:
    data_path = Path('.')

print(f"데이터 경로: {data_path.resolve()}")


# --- 파일 존재 확인 함수 ---
def check_file(path):
    """파일 존재 여부를 확인하고 결과를 반환합니다."""
    path = Path(path)
    if not path.exists():
        print(f"경고: 파일이 존재하지 않습니다 - {path}")
        return False
    print(f"파일 확인: {path}")
    return True


# --- 요소 데이터 가져오기 헬퍼 함수 ---
def get_element_data(df, condition_col, condition_val, score_col, extra_cols=[], default_score=0.0):
    """주어진 조건에 맞는 행의 점수와 추가 컬럼 값들을 가져옵니다. (NaN 처리 개선)"""
    default_result = {'score': default_score}
    for col in extra_cols:
        default_result[col] = np.nan  # 기본값은 NaN으로 설정

    if df is None or df.empty:
        return default_result
    try:
        filtered_df = df[df[condition_col] == condition_val]
        if filtered_df.empty:
            return default_result

        score_val = filtered_df[score_col].values[0]
        result = {'score': float(score_val) if pd.notna(score_val) else default_score}

        for col in extra_cols:
            if col in filtered_df.columns:
                val = filtered_df[col].values[0]
                result[col] = val if pd.notna(val) else np.nan
            else:
                result[col] = np.nan
        return result
    except Exception as e:
        print(f"Warning: Error retrieving data for {condition_val}. Error: {e}. Using defaults.")
        error_result = {'score': default_score}
        for col in extra_cols:
            error_result[col] = np.nan
        return error_result


print("데이터 경로 설정 및 헬퍼 함수 정의 완료.")


# --- 해석 출력 헬퍼 함수 ---
def summarize_candidates(candidate_df, positive_label, negative_label, top_n=2):
    """SWOT 후보 데이터에서 상/하위 핵심 항목을 요약합니다."""
    if candidate_df is None or candidate_df.empty:
        return [], []

    positive_items = candidate_df[candidate_df['swot_type'] == positive_label]
    negative_items = candidate_df[candidate_df['swot_type'] == negative_label]

    positive_items = positive_items.sort_values('score', ascending=False).head(top_n).to_dict('records')
    negative_items = negative_items.sort_values('score').head(top_n).to_dict('records')
    return positive_items, negative_items


def print_candidate_interpretation(title, candidate_df, positive_label, negative_label,
                                   positive_name, negative_name, score_unit='', top_n=2):
    """분석 결과를 사람이 읽기 쉬운 해석 문장으로 출력합니다."""
    print(f"\n[{title} 해석]")
    if candidate_df is None or candidate_df.empty:
        print("- 해석할 유의미한 SWOT 후보가 없습니다.")
        return

    positive_items, negative_items = summarize_candidates(candidate_df, positive_label, negative_label, top_n=top_n)

    if positive_items:
        for item in positive_items:
            score = item.get('score', 0)
            unit = score_unit if score_unit else ''
            print(f"- 주요 {positive_name}: {item['element']} ({score:.2f}{unit}). {item.get('detail', '')}")
    else:
        print(f"- 뚜렷한 {positive_name} 후보는 확인되지 않았습니다.")

    if negative_items:
        for item in negative_items:
            score = item.get('score', 0)
            unit = score_unit if score_unit else ''
            print(f"- 주요 {negative_name}: {item['element']} ({score:.2f}{unit}). {item.get('detail', '')}")
    else:
        print(f"- 뚜렷한 {negative_name} 후보는 확인되지 않았습니다.")



---

# Part A: 내부 분석 (강점 S / 약점 W 도출)

> 내부 데이터(판매 실적, 고객 만족도, 리뷰)를 분석하여 강점과 약점을 식별합니다.

---

## Step 4: 판매 실적 분석 — 산업 평균 대비 상대 성장률

| 항목 | 설명 |
|------|------|
| **사용 데이터** | `sales_data.csv` + `industry_growth.csv` |
| **분석 방법** | 카테고리별 성장률을 계산한 뒤 산업 평균 성장률과 비교 |
| **판단 기준** | 상대 성장률 > 0 → **강점(S)** / < 0 → **약점(W)** |
| **시각화** | 카테고리별 산업 평균 대비 상대 성장률 막대그래프 |

> **핵심**: 우리 회사의 성장률을 절대값이 아니라 산업 평균과 비교해 해석해야 실제 강점과 약점을 구분할 수 있습니다.


### [설명] 판매 실적 분석 (내부 - 강점/약점)
이 셀에서는 기업의 제품별 판매 실적 데이터를 불러와, 각 제품 카테고리별 성장률을 계산합니다.
- `sales_data.csv`에는 제품별 연도별 판매 실적이 저장되어 있습니다.
- `industry_growth.csv`에는 각 카테고리의 산업 평균 성장률이 포함되어 있습니다.
- 두 데이터를 결합하여, 우리 기업의 성장률이 산업 평균 대비 얼마나 높은지(강점) 또는 낮은지(약점) 분석합니다.
- 결과는 바 차트로 시각화하여 한눈에 비교할 수 있습니다.

이 분석을 통해 내부 역량(강점/약점)을 수치로 파악할 수 있습니다.

In [ ]:
# --- 판매 실적 분석 (내부 - 강점/약점) ---
print("\n--- 1. 판매 실적 분석 시작 ---")
sales_data_path = os.path.join(data_path, 'sales_data.csv')
industry_growth_path = os.path.join(data_path, 'industry_growth.csv')
sales_data = None
industry_growth = None
category_comparison = pd.DataFrame()
sales_swot_candidates = pd.DataFrame()

if check_file(sales_data_path) and check_file(industry_growth_path):
    sales_data = pd.read_csv(sales_data_path)
    print("판매 실적 데이터 로드 완료.")

    sales_growth = sales_data.groupby('category').filter(lambda x: len(x) >= 2).groupby('category').apply(
        lambda x: (x['sales'].iloc[-1] - x['sales'].iloc[0]) / x['sales'].iloc[0] * 100 if x['sales'].iloc[0] != 0 else 0
    ).reset_index(name='growth_rate')

    industry_growth = pd.read_csv(industry_growth_path)
    print("산업 평균 성장률 데이터 로드 완료.")

    category_comparison = pd.merge(sales_growth, industry_growth, on='category', how='left')
    category_comparison['industry_growth'] = category_comparison['industry_growth'].fillna(0)
    category_comparison['relative_growth'] = category_comparison['growth_rate'] - category_comparison['industry_growth']
    category_comparison['swot_type'] = category_comparison['relative_growth'].apply(
        lambda x: 'strengths' if x > 0 else ('weaknesses' if x < 0 else 'neutral')
    )

    sales_swot_candidates = category_comparison.loc[
        category_comparison['swot_type'] != 'neutral',
        ['category', 'growth_rate', 'industry_growth', 'relative_growth', 'swot_type']
    ].copy()
    sales_swot_candidates['element'] = sales_swot_candidates['category'] + ' 매출 성장률'
    sales_swot_candidates['score'] = sales_swot_candidates['relative_growth']
    sales_swot_candidates['source'] = '판매 실적 분석'
    sales_swot_candidates['detail'] = sales_swot_candidates.apply(
        lambda row: f"자사 성장률 {row['growth_rate']:.2f}% - 산업 평균 {row['industry_growth']:.2f}% = {row['relative_growth']:.2f}%p",
        axis=1,
    )
    sales_swot_candidates = sales_swot_candidates[['element', 'score', 'swot_type', 'source', 'detail']]
    print("카테고리별 상대 성장률 계산 및 SWOT 후보 생성 완료.")

    if not category_comparison.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(x='category', y='relative_growth', data=category_comparison)
        plt.axhline(y=0, color='r', linestyle='-')
        plt.title('카테고리별 산업 평균 대비 성장률')
        plt.ylabel('상대적 성장률 (%)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    if not sales_swot_candidates.empty:
        print("\n[판매 실적 분석 기반 SWOT 후보]")
        print(sales_swot_candidates.to_markdown(index=False))

    print_candidate_interpretation(
        "판매 실적 분석 기반 SWOT 후보 해석",
        sales_swot_candidates,
        positive_label="strengths",
        negative_label="weaknesses",
        positive_name="강점",
        negative_name="약점",
        score_unit="%p",
    )
else:
    print("판매 실적 또는 산업 성장률 데이터 파일을 찾을 수 없어 분석을 건너뜁니다.")

print("--- 1. 판매 실적 분석 완료 ---")


## Step 5: 고객 만족도 분석 — IPA(Importance-Performance Analysis)

| 항목 | 설명 |
|------|------|
| **사용 데이터** | `customer_satisfaction.csv` (속성별 만족도·중요도) + `industry_satisfaction.csv` (산업 평균 만족도) |
| **분석 방법** | IPA(중요도-성과 분석) — 중요도와 만족도를 2차원 매트릭스로 배치 |
| **판단 기준** | IPA 사분면: 중요도·만족도 모두 평균 이상 → **강점(S)** / 중요도 높고 만족도 평균 미만 → **약점(W)** / 나머지 → 제외 |
| **시각화** | IPA 산점도 — 점 크기=중요도, 색상=사분면 분류, 점선=평균 기준선 |

### IPA 사분면 해석

| 사분면 | 중요도 | 만족도 | 의미 | SWOT 반영 |
|--------|--------|--------|------|----------|
| 우상 | 높음 | 높음 | **핵심 강점** — 유지 강화 | **강점(S)** |
| 좌상 | 낮음 | 높음 | 과잉 투자 — 자원 재배분 검토 | 제외 |
| 우하 | 높음 | 낮음 | **핵심 약점** — 즉시 개선 필요 | **약점(W)** |
| 좌하 | 낮음 | 낮음 | 낮은 우선순위 | 제외 |

> **핵심**: 중요도가 높은 속성만 SWOT에 반영합니다. 중요도가 높은데 만족도가 낮은 속성(우하 사분면)은 고객이 가장 불만을 느끼는 **핵심 약점**이며, 중요도와 만족도가 모두 높은 속성(우상 사분면)은 **핵심 강점**입니다.

### [설명] 고객 만족도 분석 (내부 - 강점/약점)
이 셀에서는 고객 만족도 데이터를 활용하여, 우리 기업의 서비스나 제품이 고객에게 얼마나 긍정적으로 평가받는지 분석합니다.
- `customer_satisfaction.csv`에는 속성별 고객 만족도와 중요도가 포함되어 있습니다.
- `industry_satisfaction.csv`에는 산업 평균 만족도가 포함되어 있습니다.
- IPA 사분면(중요도·만족도 평균 기준선)으로 속성을 분류합니다.
  - 중요도·만족도 모두 높음 → 핵심 강점 (유지 강화)
  - 중요도 높고 만족도 낮음 → 핵심 약점 (즉시 개선)
  - 나머지 → SWOT 대상에서 제외 (과잉 투자 또는 낮은 우선순위)
- 산업 평균 만족도는 참고 지표로 함께 표시됩니다.

이 분석을 통해 고객이 중요하게 여기는 속성 중 잘하는 것(강점)과 못하는 것(약점)을 구체적으로 도출할 수 있습니다.

In [ ]:
# --- 고객 만족도 분석 (내부 - 강점/약점) ---
print("\n--- 2. 고객 만족도 분석 시작 ---")
satisfaction_data_path = os.path.join(data_path, 'customer_satisfaction.csv')
industry_satisfaction_path = os.path.join(data_path, 'industry_satisfaction.csv')
satisfaction_data = None
industry_satisfaction = None
satisfaction_comparison = pd.DataFrame()
satisfaction_swot_candidates = pd.DataFrame()

if check_file(satisfaction_data_path) and check_file(industry_satisfaction_path):
    satisfaction_data = pd.read_csv(satisfaction_data_path)
    print("고객 만족도 데이터 로드 완료.")

    satisfaction_matrix = satisfaction_data[['attribute', 'satisfaction_score', 'importance_score']]

    industry_satisfaction = pd.read_csv(industry_satisfaction_path)
    print("산업 평균 만족도 데이터 로드 완료.")

    satisfaction_comparison = pd.merge(satisfaction_matrix, industry_satisfaction, on='attribute', how='left')
    satisfaction_comparison['satisfaction_gap'] = satisfaction_comparison['satisfaction_score'] - satisfaction_comparison['industry_avg']

    # IPA 사분면 기준선 (각 축의 평균)
    mean_importance = satisfaction_comparison['importance_score'].mean()
    mean_satisfaction = satisfaction_comparison['satisfaction_score'].mean()

    # IPA 사분면 기반 SWOT 분류
    def classify_ipa(row):
        high_imp = row['importance_score'] >= mean_importance
        high_sat = row['satisfaction_score'] >= mean_satisfaction
        if high_imp and high_sat:
            return 'strengths'      # 우상: 핵심 강점 — 유지 강화
        elif high_imp and not high_sat:
            return 'weaknesses'     # 우하: 핵심 약점 — 즉시 개선
        else:
            return 'neutral'        # 좌상: 과잉 투자 / 좌하: 낮은 우선순위

    satisfaction_comparison['swot_type'] = satisfaction_comparison.apply(classify_ipa, axis=1)

    # IPA 사분면 라벨 (시각화·해석용)
    def get_quadrant_label(row):
        high_imp = row['importance_score'] >= mean_importance
        high_sat = row['satisfaction_score'] >= mean_satisfaction
        if high_imp and high_sat:
            return '핵심 강점 (유지 강화)'
        elif high_imp and not high_sat:
            return '핵심 약점 (즉시 개선)'
        elif not high_imp and high_sat:
            return '과잉 투자 (자원 재배분)'
        else:
            return '낮은 우선순위'

    satisfaction_comparison['quadrant'] = satisfaction_comparison.apply(get_quadrant_label, axis=1)

    # 우선순위 점수: 평균 기준선 대비 편차의 곱 (강점은 양수, 약점은 음수)
    satisfaction_comparison['priority_score'] = (
        (satisfaction_comparison['importance_score'] - mean_importance)
        * (satisfaction_comparison['satisfaction_score'] - mean_satisfaction)
    )

    satisfaction_swot_candidates = satisfaction_comparison.loc[
        satisfaction_comparison['swot_type'] != 'neutral',
        ['attribute', 'satisfaction_score', 'importance_score', 'industry_avg',
         'satisfaction_gap', 'priority_score', 'swot_type', 'quadrant']
    ].copy()
    satisfaction_swot_candidates['element'] = satisfaction_swot_candidates['attribute'] + ' 만족도'
    satisfaction_swot_candidates['score'] = satisfaction_swot_candidates['priority_score']
    satisfaction_swot_candidates['source'] = '고객 만족도 분석'
    satisfaction_swot_candidates['detail'] = satisfaction_swot_candidates.apply(
        lambda row: (
            f"IPA {row['quadrant']} | "
            f"만족도 {row['satisfaction_score']:.2f}(산업 {row['industry_avg']:.2f}), "
            f"중요도 {row['importance_score']:.2f}"
        ),
        axis=1,
    )
    satisfaction_swot_candidates = satisfaction_swot_candidates[['element', 'score', 'swot_type', 'source', 'detail']]
    print("IPA 사분면 기반 SWOT 분류 및 우선순위 점수 계산 완료.")
    print(f"  기준선: 중요도 평균 = {mean_importance:.2f}, 만족도 평균 = {mean_satisfaction:.2f}")

    if not satisfaction_comparison.dropna(subset=['satisfaction_score', 'importance_score']).empty:
        plt.figure(figsize=(10, 8))
        temp_df = satisfaction_comparison.dropna(subset=['satisfaction_score', 'importance_score']).copy()

        quadrant_colors = {
            '핵심 강점 (유지 강화)': 'green',
            '핵심 약점 (즉시 개선)': 'red',
            '과잉 투자 (자원 재배분)': 'orange',
            '낮은 우선순위': 'gray',
        }

        sns.scatterplot(
            x='importance_score',
            y='satisfaction_score',
            size='importance_score',
            hue='quadrant',
            palette=quadrant_colors,
            data=temp_df,
            legend='brief',
            sizes=(50, 500)
        )
        plt.axhline(y=mean_satisfaction, color='gray', linestyle='--', alpha=0.7,
                     label=f'만족도 평균 ({mean_satisfaction:.2f})')
        plt.axvline(x=mean_importance, color='gray', linestyle='--', alpha=0.7,
                     label=f'중요도 평균 ({mean_importance:.2f})')

        texts = []
        for _, row in temp_df.iterrows():
            texts.append(plt.text(row['importance_score'], row['satisfaction_score'], row['attribute'], fontsize=9))

        plt.title('중요도-만족도 분석 (IPA)')
        plt.xlabel('중요도')
        plt.ylabel('만족도')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='IPA 사분면', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("IPA 시각화를 위한 유효 데이터가 부족합니다.")

    if not satisfaction_swot_candidates.empty:
        print("\n[고객 만족도 분석 기반 SWOT 후보]")
        print(satisfaction_swot_candidates.to_markdown(index=False))

    print_candidate_interpretation(
        "고객 만족도 분석 기반 SWOT 후보 해석",
        satisfaction_swot_candidates,
        positive_label="strengths",
        negative_label="weaknesses",
        positive_name="강점",
        negative_name="약점",
        score_unit="점",
    )
else:
    print("고객 만족도 또는 산업 평균 만족도 데이터 파일을 찾을 수 없어 분석을 건너뜁니다.")

print("--- 2. 고객 만족도 분석 완료 ---")

---

# Part B: 외부 분석 (기회 O / 위협 T 도출)

> 외부 데이터(시장, 경쟁, 트렌드)를 분석하여 기회와 위협 요인을 식별합니다.

---

## Step 6: 시장 데이터 분석 — 매력도-경쟁 강도 매트릭스

| 항목 | 설명 |
|------|------|
| **사용 데이터** | `market_data.csv` (카테고리별 시장 규모, 성장률, 경쟁 강도) |
| **분석 방법** | 시장 매력도(성장률) × 경쟁 강도 매트릭스 분석 |
| **판단 기준** | 높은 성장률 + 낮은 경쟁 → **기회(O)** / 낮은 성장률 + 높은 경쟁 → **위협(T)** |
| **시각화** | 산점도 — X축=경쟁 강도, Y축=시장 성장률, 점 크기=시장 규모 |

### 사분면 해석

| 사분면 | 성장률 | 경쟁 강도 | 의미 |
|--------|--------|----------|------|
| 좌상 | 높음 | 낮음 | **최우선 기회** — 진입/확대 |
| 우상 | 높음 | 높음 | 매력적이나 경쟁 치열 — 차별화 필요 |
| 좌하 | 낮음 | 낮음 | 니치 시장 — 선택적 투자 |
| 우하 | 낮음 | 높음 | **위협** — 축소/철수 검토 |

> **핵심**: 시장 규모가 크고 성장률이 높으면서 경쟁이 덜한 영역이 최우선 **기회** 시장입니다.

### [설명] 시장 데이터 분석 (외부 - 기회/위협)
이 셀에서는 시장 데이터(`market_data.csv`)를 활용하여, 외부 환경에서의 기회와 위협 요인을 분석합니다.
- 시장 성장률, 경쟁 강도, 시장 규모 등의 지표를 시각화하여 각 카테고리별로 시장의 매력도와 경쟁 상황을 파악합니다.
- 성장률이 높고 경쟁이 약한 시장은 기회(Opportunity), 경쟁이 심하거나 성장률이 낮은 시장은 위협(Threat)으로 분류할 수 있습니다.

이 분석을 통해 외부 환경에서 우리 기업이 주목해야 할 기회와 위협을 도출할 수 있습니다.

In [ ]:
# --- 시장 데이터 분석 (외부 - 기회/위협) ---
print("\n--- 3. 시장 데이터 분석 시작 ---")
market_data_path = os.path.join(data_path, 'market_data.csv')
market_data = None
market_swot_candidates = pd.DataFrame()

if check_file(market_data_path):
    market_data = pd.read_csv(market_data_path)
    print("시장 데이터 로드 완료.")

    market_data['growth_norm'] = (market_data['market_growth'] - market_data['market_growth'].min()) / (
        market_data['market_growth'].max() - market_data['market_growth'].min()
    )
    market_data['competition_norm'] = (market_data['competition_intensity'] - market_data['competition_intensity'].min()) / (
        market_data['competition_intensity'].max() - market_data['competition_intensity'].min()
    )
    market_data['market_priority_score'] = market_data['growth_norm'] - market_data['competition_norm']
    market_data['swot_type'] = market_data['market_priority_score'].apply(
        lambda x: 'opportunities' if x > 0 else ('threats' if x < 0 else 'neutral')
    )

    market_swot_candidates = market_data.loc[
        market_data['swot_type'] != 'neutral',
        ['category', 'market_growth', 'competition_intensity', 'market_priority_score', 'swot_type']
    ].copy()
    market_swot_candidates['element'] = market_swot_candidates['category'] + ' 시장 환경'
    market_swot_candidates['score'] = market_swot_candidates['market_priority_score']
    market_swot_candidates['source'] = '시장 데이터 분석'
    market_swot_candidates['detail'] = market_swot_candidates.apply(
        lambda row: f"성장률 {row['market_growth']:.2f}, 경쟁 강도 {row['competition_intensity']:.2f} 기반 시장 우선순위 점수 {row['market_priority_score']:.2f}",
        axis=1,
    )
    market_swot_candidates = market_swot_candidates[['element', 'score', 'swot_type', 'source', 'detail']]
    print("시장 성장성과 경쟁 강도를 반영한 SWOT 후보 생성 완료.")

    if not market_data.dropna(subset=['market_growth', 'competition_intensity']).empty:
        plt.figure(figsize=(10, 8))
        temp_df = market_data.dropna(subset=['market_growth', 'competition_intensity', 'market_size']).copy()
        mean_competition = temp_df['competition_intensity'].mean()
        mean_growth = temp_df['market_growth'].mean()

        sns.scatterplot(
            x='competition_intensity',
            y='market_growth',
            size='market_size',
            hue='market_growth',
            palette='YlGnBu',
            data=temp_df,
            legend='brief',
            sizes=(50, 500)
        )
        plt.axhline(y=mean_growth, color='gray', linestyle='--')
        plt.axvline(x=mean_competition, color='gray', linestyle='--')

        texts = []
        for _, row in temp_df.iterrows():
            texts.append(plt.text(row['competition_intensity'], row['market_growth'], row['category'], fontsize=9))

        plt.title('시장 매력도-경쟁 강도 분석')
        plt.xlabel('경쟁 강도')
        plt.ylabel('시장 성장률(%)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='시장 성장률(%)', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("시장 매력도 시각화를 위한 유효 데이터가 부족합니다.")

    if not market_swot_candidates.empty:
        print("\n[시장 데이터 분석 기반 SWOT 후보]")
        print(market_swot_candidates.to_markdown(index=False))

    print_candidate_interpretation(
        "시장 데이터 분석 기반 SWOT 후보 해석",
        market_swot_candidates,
        positive_label="opportunities",
        negative_label="threats",
        positive_name="기회",
        negative_name="위협",
    )
else:
    print("시장 데이터 파일을 찾을 수 없어 분석을 건너뜁니다.")

print("--- 3. 시장 데이터 분석 완료 ---")


## Step 7: 소비자 트렌드 분석 — 영향력-관련성 매트릭스

| 항목 | 설명 |
|------|------|
| **사용 데이터** | `consumer_trends.csv` (트렌드명, 영향력 점수, 확산율, 비즈니스 관련성) |
| **분석 방법** | 영향력(impact_score) × 관련성(relevance) 매트릭스로 트렌드를 분류합니다 |
| **판단 기준** | 영향력 > 0 & 높은 관련성 → **기회(O)** / 영향력 < 0 & 높은 관련성 → **위협(T)** |
| **시각화** | 산점도 — X축=비즈니스 관련성, Y축=영향력, 점 크기=확산 속도, 색상=긍정(초록)/부정(빨강) |

### 트렌드 분류 기준

| 영역 | 조건 | 의미 |
|------|------|------|
| 우상 | 관련성 > 평균 & 영향력 > 0 | **핵심 기회** — 적극 활용 |
| 우하 | 관련성 > 평균 & 영향력 < 0 | **핵심 위협** — 대응 필요 |
| 좌상/좌하 | 관련성 < 평균 | 모니터링 대상 (당장 영향 적음) |

> **핵심**: 확산 속도(adoption_rate)가 빠른 트렌드일수록 대응 시급성이 높습니다. `adjustText`로 레이블 겹침을 자동 보정합니다.

### [설명] 소비자 트렌드 분석 (외부 - 기회/위협)
이 셀에서는 소비자 트렌드 데이터를 분석하여, 시장의 변화와 새로운 기회 또는 위협 요인을 파악합니다.
- `consumer_trends.csv`에는 트렌드별 영향력, 확산 속도, 비즈니스 관련성 등이 포함되어 있습니다.
- 트렌드의 영향력(긍정/부정)과 비즈니스 관련성을 기준으로 기회와 위협을 분류합니다.
- 시각화를 통해 어떤 트렌드가 우리 기업에 중요한지 한눈에 파악할 수 있습니다.

이 분석을 통해 외부 환경 변화에 능동적으로 대응할 수 있는 전략적 인사이트를 얻을 수 있습니다.

In [ ]:
# --- 소비자 트렌드 분석 (외부 - 기회/위협) ---
print("\n--- 4. 소비자 트렌드 분석 시작 ---")
trend_data_path = os.path.join(data_path, 'consumer_trends.csv')
trend_data = None
trend_analysis = pd.DataFrame()
trend_swot_candidates = pd.DataFrame()

if check_file(trend_data_path):
    trend_data = pd.read_csv(trend_data_path)
    print("소비자 트렌드 데이터 로드 완료.")

    trend_analysis = trend_data[['trend_name', 'impact_score', 'adoption_rate', 'relevance_to_business']].copy()
    trend_analysis['impact_type'] = trend_analysis['impact_score'].apply(
        lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral')
    )
    trend_analysis['trend_priority_score'] = trend_analysis['impact_score'] * trend_analysis['relevance_to_business']
    trend_analysis['swot_type'] = trend_analysis['trend_priority_score'].apply(
        lambda x: 'opportunities' if x > 0 else ('threats' if x < 0 else 'neutral')
    )

    trend_swot_candidates = trend_analysis.loc[
        trend_analysis['swot_type'] != 'neutral',
        ['trend_name', 'impact_score', 'adoption_rate', 'relevance_to_business', 'trend_priority_score', 'swot_type']
    ].copy()
    trend_swot_candidates['element'] = trend_swot_candidates['trend_name'] + ' 트렌드'
    trend_swot_candidates['score'] = trend_swot_candidates['trend_priority_score']
    trend_swot_candidates['source'] = '소비자 트렌드 분석'
    trend_swot_candidates['detail'] = trend_swot_candidates.apply(
        lambda row: f"영향력 {row['impact_score']:.2f} x 관련성 {row['relevance_to_business']:.2f} = {row['trend_priority_score']:.2f}",
        axis=1,
    )
    trend_swot_candidates = trend_swot_candidates[['element', 'score', 'swot_type', 'source', 'detail']]
    print("트렌드 영향 유형 분류 및 SWOT 후보 생성 완료.")

    if not trend_analysis.dropna(subset=['impact_score', 'relevance_to_business']).empty:
        plt.figure(figsize=(12, 8))
        temp_df = trend_analysis.dropna(subset=['impact_score', 'relevance_to_business', 'adoption_rate']).copy()
        mean_relevance = temp_df['relevance_to_business'].mean()

        sns.scatterplot(
            x='relevance_to_business',
            y='impact_score',
            size='adoption_rate',
            hue='impact_type',
            palette={'Positive': 'green', 'Negative': 'red', 'Neutral': 'grey'},
            sizes=(50, 400),
            data=temp_df,
            legend='brief'
        )
        plt.axhline(y=0, color='gray', linestyle='--')
        plt.axvline(x=mean_relevance, color='gray', linestyle='--')

        texts_trend = []
        for _, row in temp_df.iterrows():
            texts_trend.append(
                plt.text(row['relevance_to_business'], row['impact_score'], row['trend_name'], fontsize=9)
            )
        adjust_text(texts_trend, arrowprops=dict(arrowstyle='->', color='black', lw=0.5))

        plt.title('소비자 트렌드 영향 분석')
        plt.xlabel('비즈니스 관련성')
        plt.ylabel('영향력 점수(긍정/부정)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(title='영향 유형', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("트렌드 영향력 시각화를 위한 유효 데이터가 부족합니다.")

    if not trend_swot_candidates.empty:
        print("\n[소비자 트렌드 분석 기반 SWOT 후보]")
        print(trend_swot_candidates.to_markdown(index=False))

    print_candidate_interpretation(
        "소비자 트렌드 분석 기반 SWOT 후보 해석",
        trend_swot_candidates,
        positive_label="opportunities",
        negative_label="threats",
        positive_name="기회",
        negative_name="위협",
    )
else:
    print("소비자 트렌드 데이터 파일을 찾을 수 없어 분석을 건너뜁니다.")

print("--- 4. 소비자 트렌드 분석 완료 ---")


---

# Part C: 텍스트 마이닝으로 강점/약점 키워드 추출

---

## Step 8: 고객 리뷰 텍스트 마이닝 (TF-IDF + NMF)

이 셀은 **정성적 데이터(고객 리뷰)**에서 강점/약점 키워드를 자동 추출하는 핵심 분석 단계입니다.

| 항목 | 설명 |
|------|------|
| **사용 데이터** | `customer_reviews.csv` (리뷰 텍스트 + rating) |
| **분석 파이프라인** | ① rating 기반 감성 분류 → ② 한국어 형태소 분석 → ③ TF-IDF 벡터화 → ④ NMF 토픽 추출 |

### 세부 처리 과정

| 단계 | 기법 | 설명 |
|------|------|------|
| **감성 분류** | rating ≥ 4 → 긍정 / < 4 → 부정 | 리뷰를 강점/약점 그룹으로 분리 |
| **전처리** | 정규식 + 불용어 제거 | 특수문자·숫자 제거, '정말', '너무' 등 의미 없는 단어 필터링 |
| **토큰화** | `Okt.pos()` (KoNLPy) | 한국어 형태소 분석 → 2음절 이상 **명사**만 추출 (정규화+어간 추출 적용) |
| **벡터화** | `TF-IDF` (max 1000 features) | 각 리뷰에서 단어의 상대적 중요도를 0~1 점수로 변환 |
| **토픽 추출** | `NMF` (5 topics) | TF-IDF 행렬을 분해하여 자주 함께 등장하는 단어 그룹(토픽)을 발견 |

### 출력 결과의 의미

| 출력 | SWOT 역할 |
|------|----------|
| 긍정 리뷰 토픽 (5개) | **강점(S)** 키워드 그룹 (예: 배송, 품질, 디자인 등) |
| 부정 리뷰 토픽 (5개) | **약점(W)** 키워드 그룹 (예: 가격, 반품, 불량 등) |
| 카테고리별 긍정 비율 | 카테고리 간 강점/약점 비교 |
| 트렌드 매트릭스 (재시각화) | 기회/위협의 시각적 확인 |

> **핵심**: 수천 건의 리뷰를 사람이 읽지 않아도, NMF 토픽 모델링이 자동으로 핵심 강점/약점 테마를 요약해 줍니다.
> TF-IDF는 "이 단어가 이 리뷰에서 얼마나 특별한가"를, NMF는 "어떤 단어들이 같은 맥락에서 등장하는가"를 찾습니다.

### [설명] 고객 리뷰 텍스트 분석 (내부 - 강점/약점)
이 셀에서는 고객 리뷰 데이터를 자연어 처리 기법으로 분석하여, 제품/서비스의 강점과 약점을 텍스트 기반으로 도출합니다.
- `customer_reviews.csv`에는 고객의 리뷰 텍스트와 평점(rating)이 포함되어 있습니다.
- 평점에 따라 긍정/부정 리뷰를 분류하고, 한글 형태소 분석기(Okt)와 TF-IDF, NMF 기법을 활용해 주요 키워드(토픽)를 추출합니다.
- 긍정 리뷰에서는 강점, 부정 리뷰에서는 약점 관련 키워드를 도출합니다.

이 분석을 통해 수치 데이터로는 알기 어려운 고객의 생생한 의견을 SWOT 분석에 반영할 수 있습니다.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from konlpy.tag import Okt
import re

# 한국어 형태소 분석기 초기화
okt = Okt()

# 한글 불용어 리스트 (예시)
korean_stopwords = [
    '정말', '너무', '같습니다', '있습니다', '합니다', '것이', '입니다', '에서', '으로', '에게',
    '제품', '점', '다음', '다른', '비교', '확실히', '같아요', '아니에요', '별로예요', '좀'
]

# 한글 텍스트 전처리 함수
def preprocess_text(text):
    text = re.sub(r'[^가-힣\s]', '', text)
    text = ' '.join(text.split())
    return text

# 키워드 추출 함수
def extract_keywords(text):
    text = preprocess_text(text)
    tokens = okt.pos(text, norm=True, stem=True)
    keywords = [
        token for token, pos in tokens
        if pos in ['Noun'] and len(token) >= 2 and token not in korean_stopwords
    ]
    return keywords

reviews_data = pd.read_csv(Path(data_path) / 'customer_reviews.csv')
reviews_data['sentiment'] = reviews_data['rating'].apply(
    lambda x: 'positive' if x >= 4 else 'negative'
)

print("감성 분석 결과 (Rating 기반):")
print(reviews_data['sentiment'].value_counts())

category_sentiment = reviews_data.groupby(['product_category', 'sentiment']).size().unstack(fill_value=0)
category_sentiment['total_reviews'] = category_sentiment.sum(axis=1)
category_sentiment['positive_ratio'] = category_sentiment['positive'] / category_sentiment['total_reviews']
overall_positive_ratio = reviews_data['sentiment'].eq('positive').sum() / len(reviews_data)
category_sentiment['review_score'] = category_sentiment['positive_ratio'] - overall_positive_ratio
category_sentiment = category_sentiment.sort_values('review_score', ascending=False)

review_swot_candidates = category_sentiment.reset_index()[['product_category', 'positive_ratio', 'review_score']].copy()
review_swot_candidates['swot_type'] = review_swot_candidates['review_score'].apply(
    lambda x: 'strengths' if x > 0 else ('weaknesses' if x < 0 else 'neutral')
)
review_swot_candidates = review_swot_candidates[review_swot_candidates['swot_type'] != 'neutral'].copy()
review_swot_candidates['element'] = review_swot_candidates['product_category'] + ' 고객 리뷰 반응'
review_swot_candidates['score'] = review_swot_candidates['review_score']
review_swot_candidates['source'] = '고객 리뷰 텍스트 분석'
review_swot_candidates['detail'] = review_swot_candidates.apply(
    lambda row: f"카테고리 긍정 비율 {row['positive_ratio']:.2%} - 전체 평균 {overall_positive_ratio:.2%} = {row['review_score']:.2%}p",
    axis=1,
)
review_swot_candidates = review_swot_candidates[['element', 'score', 'swot_type', 'source', 'detail']]

plt.figure(figsize=(12, 6))
sns.barplot(x=category_sentiment.index, y=category_sentiment['positive_ratio'])
plt.axhline(y=overall_positive_ratio, color='gray', linestyle='--', label='전체 평균 긍정 비율')
plt.title('카테고리별 긍정 리뷰 비율')
plt.xlabel('제품 카테고리')
plt.ylabel('긍정 리뷰 비율')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

vectorizer = TfidfVectorizer(max_features=1000, tokenizer=extract_keywords)
positive_reviews = reviews_data[reviews_data['sentiment'] == 'positive']['review_text']
negative_reviews = reviews_data[reviews_data['sentiment'] == 'negative']['review_text']

strengths_topics = []
weaknesses_topics = []

if len(positive_reviews) >= 5:
    tfidf_positive = vectorizer.fit_transform(positive_reviews)
    nmf_positive = NMF(n_components=5, random_state=42)
    nmf_positive.fit_transform(tfidf_positive)
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(nmf_positive.components_):
        top_features_idx = topic.argsort()[:-11:-1]
        top_features = [feature_names[i] for i in top_features_idx]
        strengths_topics.append(f"강점 토픽 {topic_idx + 1}: {', '.join(top_features)}")

if len(negative_reviews) >= 5:
    tfidf_negative = vectorizer.fit_transform(negative_reviews)
    nmf_negative = NMF(n_components=5, random_state=42)
    nmf_negative.fit_transform(tfidf_negative)
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(nmf_negative.components_):
        top_features_idx = topic.argsort()[:-11:-1]
        top_features = [feature_names[i] for i in top_features_idx]
        weaknesses_topics.append(f"약점 토픽 {topic_idx + 1}: {', '.join(top_features)}")

print("\n강점 관련 주요 토픽:")
for topic in strengths_topics:
    print(topic)

print("\n약점 관련 주요 토픽:")
for topic in weaknesses_topics:
    print(topic)

if not review_swot_candidates.empty:
    print("\n[고객 리뷰 텍스트 분석 기반 SWOT 후보]")
    print(review_swot_candidates.to_markdown(index=False))

print_candidate_interpretation(
    "고객 리뷰 텍스트 분석 기반 SWOT 후보 해석",
    review_swot_candidates,
    positive_label="strengths",
    negative_label="weaknesses",
    positive_name="강점",
    negative_name="약점",
    score_unit="%p",
)


---

# Part D: SWOT 통합 및 전략 수립

---

## Step 9: SWOT 매트릭스 통합 및 계량화

이 단계에서는 앞선 5개 분석 결과를 공통 규칙으로 변환한 뒤, 중복 없이 최종 SWOT 요소를 자동 선정합니다.

| 분석 단계 | 입력 지표 | 통합용 점수 | 분류 기준 |
|------|------|------|------|
| Step 4 판매 실적 | `relative_growth` | 자사 성장률 - 산업 평균 성장률 | `+` → S / `-` → W |
| Step 5 고객 만족도 | `satisfaction_gap`, `importance_score` | `satisfaction_gap × importance_score` | `+` → S / `-` → W |
| Step 8 리뷰 분석 | `positive_ratio` | 카테고리 긍정 비율 - 전체 평균 긍정 비율 | `+` → S / `-` → W |
| Step 6 시장 데이터 | `market_growth`, `competition_intensity` | 정규화 성장률 - 정규화 경쟁 강도 | `+` → O / `-` → T |
| Step 7 트렌드 분석 | `impact_score`, `relevance_to_business` | `impact_score × relevance_to_business` | `+` → O / `-` → T |

### 통합 절차

1. 각 분석은 `element`, `score`, `swot_type`, `source`, `detail` 컬럼을 가진 **SWOT 후보 표**를 하나씩 생성합니다.
2. Step 9에서는 이 후보 표들을 내부용과 외부용으로 합칩니다.
3. 같은 항목이 반복되면 점수 절대값이 큰 항목만 남겨 **중복을 제거**합니다.
4. 강점/기회는 점수가 큰 순서대로, 약점/위협은 절대값이 큰 순서대로 상위 항목만 선택합니다.
5. 이렇게 선정된 항목만 최종 `swot_elements`에 들어가고, 이후 TOWS 전략 생성에 사용됩니다.

> **핵심**: Step 9는 사람이 예시 항목을 수동 입력하는 단계가 아니라, 앞선 분석 결과를 공통 기준으로 합쳐 최종 SWOT를 자동 구성하는 단계입니다.


### [설명] SWOT 요소 정리 및 시각화
이 셀에서는 앞서 분석한 내부(강점/약점)와 외부(기회/위협) 데이터를 바탕으로, 주요 SWOT 요소를 정리하고 시각화합니다.
- 각 요소별로 대표적인 항목과 점수를 정리하여 한눈에 볼 수 있도록 합니다.
- 바 차트로 시각화하여 강점, 약점, 기회, 위협의 상대적 크기를 비교할 수 있습니다.

이 과정을 통해 SWOT 분석 결과를 직관적으로 파악할 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


def deduplicate_candidates(df):
    if df.empty:
        return df
    ordered = df.copy()
    ordered['abs_score'] = ordered['score'].abs()
    ordered = ordered.sort_values('abs_score', ascending=False)
    ordered = ordered.drop_duplicates(subset=['element'], keep='first')
    return ordered.drop(columns=['abs_score'])


def build_swot_elements(internal_candidates, external_candidates, top_n=4):
    swot_elements = {
        'strengths': [],
        'weaknesses': [],
        'opportunities': [],
        'threats': [],
    }

    if not internal_candidates.empty:
        strengths_df = internal_candidates[internal_candidates['swot_type'] == 'strengths'].sort_values('score', ascending=False).head(top_n)
        weaknesses_df = internal_candidates[internal_candidates['swot_type'] == 'weaknesses'].sort_values('score').head(top_n)
        swot_elements['strengths'] = strengths_df[['element', 'score', 'source', 'detail']].to_dict('records')
        swot_elements['weaknesses'] = weaknesses_df[['element', 'score', 'source', 'detail']].to_dict('records')

    if not external_candidates.empty:
        opportunities_df = external_candidates[external_candidates['swot_type'] == 'opportunities'].sort_values('score', ascending=False).head(top_n)
        threats_df = external_candidates[external_candidates['swot_type'] == 'threats'].sort_values('score').head(top_n)
        swot_elements['opportunities'] = opportunities_df[['element', 'score', 'source', 'detail']].to_dict('records')
        swot_elements['threats'] = threats_df[['element', 'score', 'source', 'detail']].to_dict('records')

    return swot_elements


internal_candidate_frames = [
    df for df in [sales_swot_candidates, satisfaction_swot_candidates, review_swot_candidates]
    if isinstance(df, pd.DataFrame) and not df.empty
]
external_candidate_frames = [
    df for df in [market_swot_candidates, trend_swot_candidates]
    if isinstance(df, pd.DataFrame) and not df.empty
]

internal_candidates = pd.concat(internal_candidate_frames, ignore_index=True) if internal_candidate_frames else pd.DataFrame(columns=['element', 'score', 'swot_type', 'source', 'detail'])
external_candidates = pd.concat(external_candidate_frames, ignore_index=True) if external_candidate_frames else pd.DataFrame(columns=['element', 'score', 'swot_type', 'source', 'detail'])

internal_candidates = deduplicate_candidates(internal_candidates)
external_candidates = deduplicate_candidates(external_candidates)

print("[내부 분석 통합 후보]")
if internal_candidates.empty:
    print("내부 후보가 없습니다.")
else:
    print(internal_candidates.sort_values('score', ascending=False).to_markdown(index=False))

print("\n[외부 분석 통합 후보]")
if external_candidates.empty:
    print("외부 후보가 없습니다.")
else:
    print(external_candidates.sort_values('score', ascending=False).to_markdown(index=False))

swot_elements = build_swot_elements(internal_candidates, external_candidates, top_n=4)

print("\n[최종 SWOT 요소]")
for category, items in swot_elements.items():
    print(f"\n- {category}")
    if not items:
        print("  선택된 항목 없음")
        continue
    print(pd.DataFrame(items).to_markdown(index=False))



def print_swot_summary(swot_elements):
    print("\n[통합 SWOT 해석]")
    label_map = {
        'strengths': '강점',
        'weaknesses': '약점',
        'opportunities': '기회',
        'threats': '위협',
    }
    for category, label in label_map.items():
        items = swot_elements.get(category, [])
        if items:
            top_item = items[0]
            print(f"- 핵심 {label}: {top_item['element']} ({top_item['score']:.2f}). {top_item.get('detail', '')}")
        else:
            print(f"- {label} 항목은 이번 통합 결과에서 선정되지 않았습니다.")

plt.figure(figsize=(14, 10))
gs = plt.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.4, hspace=0.3)
colors = {
    'strengths': 'green',
    'weaknesses': 'red',
    'opportunities': 'blue',
    'threats': 'orange'
}

for plot_idx, category in enumerate(['strengths', 'weaknesses', 'opportunities', 'threats']):
    row = plot_idx // 2
    col = plot_idx % 2
    ax = plt.subplot(gs[row, col])
    elements = swot_elements[category]
    if not elements:
        ax.set_title(f'{category.title()} (데이터 없음)')
        ax.axis('off')
        continue

    sort_reverse = category in ['strengths', 'opportunities']
    elements = sorted(elements, key=lambda x: x['score'], reverse=sort_reverse)
    labels = [e['element'] for e in elements]
    scores = [abs(e['score']) if category in ['weaknesses', 'threats'] else e['score'] for e in elements]
    y_pos = range(len(elements))

    ax.barh(y_pos, scores, color=colors[category], alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_title(f"{category.title()} ({'강점' if category == 'strengths' else '약점' if category == 'weaknesses' else '기회' if category == 'opportunities' else '위협'})", fontweight='bold', color=colors[category])
    ax.set_xlim(0, max(scores) * 1.2 if max(scores) > 0 else 1)
    for i, v in enumerate(scores):
        ax.text(v + (max(scores) * 0.03 if max(scores) > 0 else 0.03), i, f"{v:.2f}", va='center')

plt.suptitle('데이터 기반 계량화된 SWOT 분석 - 디지털마트', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('quantified_swot_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print_swot_summary(swot_elements)


## Step 10: TOWS 전략 생성 및 우선순위화

| 항목 | 설명 |
|------|------|
| **입력** | Step 9에서 정리한 강점·약점·기회·위협 요소와 점수 |
| **분석 방법** | SO, ST, WO, WT 조합별 전략 문장을 생성하고 영향력·실현 가능성을 반영해 점수화 |
| **LLM 활용** | OpenAI API로 각 전략 유형별 실행 아이디어를 보강 |
| **출력** | 전략 목록, 전략 유형, 영향력, 실현 가능성, 우선순위 점수 |

### [설명] TOWS 전략 생성 및 LLM(OpenAI) 활용
이 셀에서는 정량화된 SWOT 요소를 바탕으로, TOWS 매트릭스 전략을 자동으로 생성합니다.
- 각 강점/약점/기회/위협 요소를 조합하여 SO, ST, WO, WT 전략을 도출합니다.
- OpenAI LLM(GPT)을 활용해 각 전략에 대한 구체적인 실행 아이디어를 생성합니다.
- 전략별로 영향력, 실현 가능성, 우선순위 점수를 계산하여 정렬합니다.

이 과정을 통해 데이터 기반의 실질적이고 창의적인 전략을 자동으로 도출할 수 있습니다.


In [ ]:
# --- TOWS 전략 생성 및 LLM(OpenAI) 활용 ---
print("\n--- 7. TOWS 전략 생성 및 LLM(OpenAI) 활용 시작 ---")

# 1. LLM(OpenAI) API 호출 함수 (Exponential Backoff 적용)
def call_openai_llm(prompt, strategy_type, retries=3, initial_delay=5):
    """OpenAI API를 호출하여 특정 전략 타입에 대한 제안을 생성합니다. (Exponential Backoff 적용)"""
    if not openai_client:
        return f"LLM({strategy_type}) 기본 응답: API 키 없음. 프롬프트: {prompt[:50]}..."

    instruction = ""
    if strategy_type == 'SO': instruction = "이 강점과 기회를 활용하여 구체적이고 실행 가능한 '성장 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'ST': instruction = "이 강점을 활용하여 주어진 위협에 효과적으로 대응하거나 그 영향을 최소화할 수 있는 '차별화 또는 방어 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'WO': instruction = "이 기회를 활용하여 주어진 약점을 극복하거나 개선할 수 있는 '역량 강화 또는 전환 전략' 아이디어를 1~2문장으로 제안해주세요..."
    elif strategy_type == 'WT': instruction = "이 약점과 위협 상황을 고려하여 부정적 영향을 최소화하거나 회피할 수 있는 '축소, 회피, 또는 방어적 전략' 아이디어를 1~2문장으로 제안해주세요..."

    full_prompt = f"{prompt}\n\n{instruction}"
    current_delay = initial_delay

    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "당신은 비즈니스 전략 전문가입니다. 주어진 SWOT 요소를 바탕으로 간결하고 실행 가능한 전략을 한국어로 제안합니다."},
                    {"role": "user", "content": full_prompt},
                ],
                max_tokens=150,
                temperature=0.7,
            )

            if response and response.choices:
                result_text = response.choices[0].message.content.strip().replace('\n', ' ')
                print(f"    OpenAI 응답 수신 성공 (시도 {attempt+1})")
                return f"LLM({strategy_type}): {result_text}"
            else:
                print(f"    Warning (시도 {attempt+1}): OpenAI가 유효한 텍스트 응답을 반환하지 않았습니다.")
                if attempt == retries - 1:
                    return f"LLM({strategy_type}) 응답 오류: 빈 응답. 프롬프트: {prompt[:50]}..."

        except Exception as e:
            print(f"    Error (시도 {attempt+1}): OpenAI API 호출 중 오류 발생: {e}")
            if "429" in str(e) or "Rate limit" in str(e) or "rate_limit" in str(e):
                wait_time = current_delay * (2 ** attempt) + np.random.uniform(0, 1)
                print(f"      Rate limit 도달 가능성. {wait_time:.2f}초 후 재시도...")
                time.sleep(wait_time)
                if attempt == retries - 1:
                    print(f"      최대 재시도 ({retries}회) 후에도 Rate limit 오류 지속.")
                    return f"LLM({strategy_type}) 오류: Rate limit 지속 ({e}). 프롬프트: {prompt[:50]}..."
            else:
                if attempt == retries - 1:
                    return f"LLM({strategy_type}) 오류: API 호출 실패 ({e}). 프롬프트: {prompt[:50]}..."
                else:
                    print(f"      기타 오류 발생. {initial_delay / 2}초 후 재시도...")
                    time.sleep(initial_delay / 2)

    return f"LLM({strategy_type}) 오류: 최대 재시도 실패. 프롬프트: {prompt[:50]}..."


# 2. 점수 정규화 함수
def normalize_scores(swot_elements):
    print("  점수 정규화 중...")
    scaler = MinMaxScaler(feature_range=(0, 1))
    elements_copy = {cat: [item.copy() for item in items] for cat, items in swot_elements.items()}
    for category, items in elements_copy.items():
        valid_scores = [item['score'] for item in items if pd.notna(item.get('score'))]
        if not valid_scores:
            for item in items: item['norm_score'] = np.nan
            continue
        if len(valid_scores) == 1:
            norm_value = 0.5
            for item in items:
                 if pd.notna(item.get('score')): item['norm_score'] = norm_value
                 else: item['norm_score'] = np.nan
            continue
        try:
            scores_array = np.array(valid_scores).reshape(-1, 1)
            normalized_scores_map = dict(zip(valid_scores, scaler.fit_transform(scores_array).flatten()))
            for item in items:
                score = item.get('score')
                if pd.notna(score): item['norm_score'] = normalized_scores_map.get(score, np.nan)
                else: item['norm_score'] = np.nan
        except Exception as e:
            print(f"    '{category}' 정규화 중 오류: {e}. NaN으로 설정.")
            for item in items: item['norm_score'] = np.nan
    print("  점수 정규화 완료.")
    return elements_copy

# 3. 내부/외부 점수 계산 함수
def calculate_scores(normalized_swot_elements):
    print("  내부/외부 종합 점수 계산 중...")
    scores = {}
    for category, items in normalized_swot_elements.items():
        valid_norm_scores = [item['norm_score'] for item in items if pd.notna(item.get('norm_score'))]
        if not valid_norm_scores: scores[category] = 0.0
        else:
            if category in ['weaknesses', 'threats']: scores[category] = np.mean([abs(s) for s in valid_norm_scores])
            else: scores[category] = np.mean(valid_norm_scores)
    internal_score = scores.get('strengths', 0.0) - scores.get('weaknesses', 0.0)
    external_score = scores.get('opportunities', 0.0) - scores.get('threats', 0.0)
    print(f"  계산된 내부 점수: {internal_score:.2f}, 외부 점수: {external_score:.2f}")
    return internal_score, external_score

# 4. TOWS 전략 생성 함수
def generate_tows_strategies(normalized_swot_elements):
    """SWOT 요소를 조합하고 OpenAI LLM을 호출하여 TOWS 전략을 생성합니다."""
    print("  TOWS 전략 생성 및 OpenAI LLM 호출 중...")
    strategies = {'SO': [], 'ST': [], 'WO': [], 'WT': []}
    valid_elements = {cat: [item for item in items if pd.notna(item.get('norm_score'))]
                      for cat, items in normalized_swot_elements.items()}
    call_interval = 1.0
    print(f"    API 호출 간격: {call_interval} 초")

    total_calls = sum(len(valid_elements.get(c1, [])) * len(valid_elements.get(c2, []))
                      for c1, c2 in [('strengths', 'opportunities'), ('strengths', 'threats'),
                                     ('weaknesses', 'opportunities'), ('weaknesses', 'threats')])
    print(f"    예상 총 API 호출 횟수: {total_calls}")
    call_count = 0

    print("  SO 전략 생성 시작...")
    for s, o in product(valid_elements.get('strengths', []), valid_elements.get('opportunities', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) SO 조합 처리 중...")
        prompt = f"강점: {s['element']} (정규화 점수: {s['norm_score']:.2f}), 기회: {o['element']} (정규화 점수: {o['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'SO')
        strategies['SO'].append({'description': llm_response, 'impact_score': (s['norm_score'] + o['norm_score']) / 2, 'feasibility': 0.6 + 0.3 * (s['norm_score'] + o['norm_score']) / 2, 'category': 'SO'})
        time.sleep(call_interval)

    print("  ST 전략 생성 시작...")
    for s, t in product(valid_elements.get('strengths', []), valid_elements.get('threats', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) ST 조합 처리 중...")
        prompt = f"강점: {s['element']} (정규화 점수: {s['norm_score']:.2f}), 위협: {t['element']} (정규화 점수: {t['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'ST')
        strategies['ST'].append({'description': llm_response, 'impact_score': (s['norm_score'] + abs(t['norm_score'])) / 2, 'feasibility': 0.5 + 0.3 * s['norm_score'], 'category': 'ST'})
        time.sleep(call_interval)

    print("  WO 전략 생성 시작...")
    for w, o in product(valid_elements.get('weaknesses', []), valid_elements.get('opportunities', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) WO 조합 처리 중...")
        prompt = f"약점: {w['element']} (정규화 점수: {w['norm_score']:.2f}), 기회: {o['element']} (정규화 점수: {o['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'WO')
        strategies['WO'].append({'description': llm_response, 'impact_score': (abs(w['norm_score']) + o['norm_score']) / 2, 'feasibility': 0.4 + 0.3 * o['norm_score'], 'category': 'WO'})
        time.sleep(call_interval)

    print("  WT 전략 생성 시작...")
    for w, t in product(valid_elements.get('weaknesses', []), valid_elements.get('threats', [])):
        call_count += 1
        print(f"    ({call_count}/{total_calls}) WT 조합 처리 중...")
        prompt = f"약점: {w['element']} (정규화 점수: {w['norm_score']:.2f}), 위협: {t['element']} (정규화 점수: {t['norm_score']:.2f})"
        llm_response = call_openai_llm(prompt, 'WT')
        strategies['WT'].append({'description': llm_response, 'impact_score': (abs(w['norm_score']) + abs(t['norm_score'])) / 2, 'feasibility': 0.3 + 0.2 * (1 - (abs(w['norm_score']) + abs(t['norm_score'])) / 2), 'category': 'WT'})
        time.sleep(call_interval)

    print("  TOWS 전략 생성 완료.")
    return strategies

# 5. 전략 우선순위 평가 함수
def prioritize_strategies(strategies):
    print("  전략 우선순위 평가 중...")
    strategy_list = [
        {**strategy, 'priority_score': strategy.get('impact_score', 0) * strategy.get('feasibility', 0)}
        for category in strategies for strategy in strategies[category]
    ]
    if not strategy_list:
        print("    생성된 전략이 없어 우선순위를 평가할 수 없습니다.")
        return pd.DataFrame()
    strategy_df = pd.DataFrame(strategy_list)
    strategy_df = strategy_df.sort_values('priority_score', ascending=False)
    print("  전략 우선순위 평가 완료.")
    return strategy_df



def print_strategy_interpretation(strategy_df, internal_score, external_score):
    print("\n[전략 우선순위 해석]")
    if strategy_df is None or strategy_df.empty:
        print("- 생성된 전략이 없어 해석할 수 없습니다.")
        return

    if internal_score >= 0 and external_score >= 0:
        position = 'SO(공격적)'
    elif internal_score < 0 and external_score >= 0:
        position = 'WO(회복)'
    elif internal_score >= 0 and external_score < 0:
        position = 'ST(다각화/방어)'
    else:
        position = 'WT(방어적)'

    top_strategy = strategy_df.iloc[0]
    print(f"- 내부 점수 {internal_score:.2f}, 외부 점수 {external_score:.2f}로 현재 전략적 위치는 {position} 성격이 강합니다.")
    print(f"- 최우선 전략은 {top_strategy['category']} 유형이며 우선순위 점수는 {top_strategy['priority_score']:.2f}입니다.")
    print(f"- 전략 요약: {top_strategy['description']}")

# --- 실행 ---
# 1. 점수 정규화
normalized_swot = normalize_scores(swot_elements)
# 2. 내부/외부 점수 계산
internal_score, external_score = calculate_scores(normalized_swot)
# 3. TOWS 전략 생성 (OpenAI 사용, 시간 소요 예상)
generated_strategies = generate_tows_strategies(normalized_swot)
# 4. 우선순위 평가
strategy_df = prioritize_strategies(generated_strategies)
print_strategy_interpretation(strategy_df, internal_score, external_score)

print("--- 7. TOWS 전략 생성 및 LLM(OpenAI) 활용 완료 ---")

## Step 11: 최종 결과 — 전략적 위치 시각화 및 우선순위

| 항목 | 설명 |
|------|------|
| **입력** | Step 10에서 계산된 내부/외부 점수 + 우선순위가 매겨진 전략 목록 |
| **시각화** | 전략적 위치 산점도 — X축=내부 역량, Y축=외부 환경, ★=현재 위치 |
| **출력** | 우선순위 상위 5개 전략 (카테고리, 설명, 영향력, 실현가능성, 우선순위 점수) |

### 사분면별 전략 방향

| 사분면 | 위치 | 의미 | 권장 전략 |
|--------|------|------|----------|
| 우상 (1사분면) | 내부↑ 외부↑ | 강점+기회 우세 | **SO 전략** (공격적 성장) |
| 좌상 (2사분면) | 내부↓ 외부↑ | 기회는 있으나 내부 약점 | **WO 전략** (역량 보강 후 성장) |
| 우하 (4사분면) | 내부↑ 외부↓ | 강점은 있으나 외부 위협 | **ST 전략** (다각화로 위험 분산) |
| 좌하 (3사분면) | 내부↓ 외부↓ | 약점+위협 우세 | **WT 전략** (방어·축소·철수) |

> **최종 결과**: ★의 위치가 어느 사분면에 있는지에 따라 기업의 **전략적 방향**이 결정됩니다.

### [설명] 전략적 위치 시각화 및 결과 요약
이 셀에서는 도출된 전략의 우선순위와 기업의 전략적 위치를 시각화합니다.
- 내부/외부 점수를 2차원 평면에 표시하여, 우리 기업이 SO/WO/ST/WT 중 어떤 전략에 집중해야 하는지 한눈에 보여줍니다.
- 우선순위가 높은 전략 5개를 표로 요약해 실제 실행에 참고할 수 있도록 합니다.

이 과정을 통해 SWOT-TOWS 분석의 최종 결과를 시각적으로 이해하고, 실질적인 전략 수립에 활용할 수 있습니다.

In [ ]:
# --- 결과 시각화 및 출력 ---
print("\n--- 8. 최종 결과 시각화 및 요약 시작 ---")

def visualize_strategies(strategy_df, internal_score, external_score):
    """전략적 위치를 시각화하고 우선순위가 높은 전략을 출력합니다."""
    # 전략적 위치 시각화
    plt.figure(figsize=(10, 8))
    limit_x = max(abs(internal_score), 0.1) * 1.5
    limit_y = max(abs(external_score), 0.1) * 1.5

    plt.scatter(internal_score, external_score, s=300, color='C4', zorder=5, label=f'현재 위치 ({internal_score:.2f}, {external_score:.2f})', marker='*')
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    plt.text(internal_score, external_score + limit_y*0.05, f"({internal_score:.2f}, {external_score:.2f})", fontsize=10, ha='center', va='bottom')

    # 사분면 텍스트
    plt.text(limit_x*0.5, limit_y*0.5, "SO 전략\n(공격적)", fontsize=12, ha='center', va='center', color='green', alpha=0.7)
    plt.text(-limit_x*0.5, limit_y*0.5, "WO 전략\n(회복)", fontsize=12, ha='center', va='center', color='blue', alpha=0.7)
    plt.text(limit_x*0.5, -limit_y*0.5, "ST 전략\n(다각화)", fontsize=12, ha='center', va='center', color='orange', alpha=0.7)
    plt.text(-limit_x*0.5, -limit_y*0.5, "WT 전략\n(방어적)", fontsize=12, ha='center', va='center', color='red', alpha=0.7)

    plt.title('전략적 위치 분석 (TOWS)', fontsize=16, fontweight='bold')
    plt.xlabel(f'내부 역량 점수 (정규화 기반)', fontsize=12)
    plt.ylabel(f'외부 환경 점수 (정규화 기반)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xlim(-limit_x, limit_x)
    plt.ylim(-limit_y, limit_y)
    plt.legend(loc='upper right')
    plt.tight_layout()
    # plt.savefig('tows_strategic_position.png', dpi=300)
    plt.show()

    # 우선순위 전략 출력
    if not strategy_df.empty:
        print("\nLLM 기반 우선순위 전략 제안 (상위 5개):")
        # description이 너무 길 경우 줄여서 표시
        strategy_df['description_short'] = strategy_df['description'].str[:100] + '...'
        # 표시할 컬럼 선택 및 포맷팅
        display_df = strategy_df[['category', 'description_short', 'priority_score', 'impact_score', 'feasibility']].head(5)
        # 점수 소수점 2자리까지 표시
        pd.options.display.float_format = '{:.2f}'.format
        print(display_df.to_markdown(index=False))
        # 원상 복구 (선택적)
        pd.reset_option('display.float_format')


        print("\n*참고:")
        print("  - 'priority_score' = 'impact_score' * 'feasibility'. 상대적 우선순위를 나타냅니다.")
    else:
        print("\n생성된 전략이 없어 우선순위를 표시할 수 없습니다.")

    print("\n[전략적 위치 해석]")
    if internal_score >= 0 and external_score >= 0:
        print("- 내부 역량과 외부 환경이 모두 상대적으로 양호해 SO(공격적) 전략에 적합한 위치입니다.")
    elif internal_score < 0 and external_score >= 0:
        print("- 외부 기회는 존재하지만 내부 역량 보완이 필요해 WO(회복) 전략이 우선입니다.")
    elif internal_score >= 0 and external_score < 0:
        print("- 내부 강점은 있으나 외부 위협이 커서 ST(방어/차별화) 전략이 적합합니다.")
    else:
        print("- 내부 약점과 외부 위협이 동시에 커서 WT(방어적) 전략이 우선입니다.")
    print(f"- 좌표 해석: 내부 {internal_score:.2f}, 외부 {external_score:.2f}로 표시됩니다.")

# 시각화 및 결과 출력 실행
# 이전 셀에서 계산된 strategy_df, internal_score, external_score 사용
if 'strategy_df' in locals() and 'internal_score' in locals() and 'external_score' in locals():
     visualize_strategies(strategy_df, internal_score, external_score)
else:
     print("오류: 전략 데이터 또는 내부/외부 점수가 계산되지 않아 결과를 표시할 수 없습니다.")

print("\n--- 8. 최종 결과 시각화 및 요약 완료 ---")
print("\n=== 전체 프로세스 종료 ===")

# SWOT 분석 결과 해석

## 분석 과정 설명

이 노트북은 내부 분석 3개와 외부 분석 2개의 결과를 순서대로 통합하여 최종 SWOT를 구성합니다.

1. 내부 분석 통합
   - 판매 실적 분석은 `산업 평균 대비 상대 성장률`로 강점/약점을 나눕니다.
   - 고객 만족도 분석은 IPA 사분면(중요도·만족도 평균 기준)으로 분류하고, 기준선 대비 편차의 곱으로 우선순위를 산정합니다.
   - 리뷰 분석은 `카테고리 긍정 비율 - 전체 평균 긍정 비율`로 고객 반응을 비교합니다.

2. 외부 분석 통합
   - 시장 데이터 분석은 `정규화 성장률 - 정규화 경쟁 강도`로 시장 매력도를 계산합니다.
   - 소비자 트렌드 분석은 `영향력 × 비즈니스 관련성`으로 기회/위협 정도를 계산합니다.

3. 최종 SWOT 선정
   - 각 분석이 만든 SWOT 후보 표를 내부용과 외부용으로 합칩니다.
   - 같은 항목이 반복되면 절대값이 큰 점수만 남겨 중복을 제거합니다.
   - 강점/기회는 높은 점수 순, 약점/위협은 절대값이 큰 순으로 상위 항목을 뽑습니다.

## 해석 포인트

최종 SWOT는 예시 문장을 수동 입력한 결과가 아니라, 앞선 분석 셀에서 만든 후보 점수를 공통 기준으로 다시 정리한 결과입니다. 따라서 특정 항목이 강점인지 약점인지 확인하려면 Step 9의 최종 표만 볼 것이 아니라, 바로 앞 단계의 후보 표와 `detail` 컬럼을 함께 읽어야 합니다.

예를 들어 내부 강점으로 선정된 항목은 다음과 같은 방식으로 근거를 추적할 수 있습니다.
- 판매 실적 기반 강점: 자사 성장률이 산업 평균보다 얼마나 높은가
- 고객 만족도 기반 강점: 중요도와 만족도가 모두 평균 이상인 속성 (IPA 우상 사분면)
- 고객 만족도 기반 약점: 중요도가 높은데 만족도가 평균 미만인 속성 (IPA 우하 사분면)
- 리뷰 기반 강점: 특정 카테고리의 긍정 반응이 전체 평균보다 얼마나 높은가

외부 기회/위협도 같은 방식입니다.
- 시장 기반 기회: 성장성이 높고 경쟁 강도가 상대적으로 낮은가
- 트렌드 기반 기회/위협: 영향력의 방향과 비즈니스 관련성이 얼마나 큰가